# 11 — Persona cây × Lifecycle

Giữ nguyên P1–P6 của notebook 10, chỉ thêm bối cảnh lifecycle từ notebook 05. Không refit cây, không tạo persona mới. CLV và affinity là bước tiếp sau notebook này.

**Input:** output 10 đã chạy xong cell cuối (personas_tree_YYYY-MM.parquet), output 05 (lifecycle_YYYY-MM.parquet). Thêm hai nguồn vào Kaggle, chỉnh đường dẫn nếu nhiều bản. Cần pandas/numpy/pyarrow/matplotlib.

**Quan trọng:** cây chỉ gán cho khách có snapshot đầy đủ và đủ lịch sử. Vì thế New/Win-back có thể bị loại nhiều; At-risk/Churned hiện không có snapshot nên thường không có persona hiện tại. Không điền persona cũ sang khách vắng dữ liệu. Vắng snapshot là proxy hiện diện panel, không xác nhận churn ngân hàng.

Ba câu hỏi:
1. Mỗi persona đang nằm trong lifecycle nào? Các state nào bị loại khỏi cohort cây?
2. Trong cùng persona, lifecycle có tạo khác biệt acquisition tháng sau không?
3. Khách chuyển lifecycle thế nào, và coverage có cho phép kết luận không?

Đánh giá vẫn là development Dec2015–Apr2016 → Jan–May2016. Tháng 05/2016 chỉ mô tả hiện tại; không có nhãn 06/2016.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
TREE_ROOT=None
LIFECYCLE_ROOT=None
EVAL_MONTHS=['2015-12','2016-01','2016-02','2016-03','2016-04']
RECOMMEND_MONTH='2016-05'
MIN_N=1000;MIN_POS=20
STATES=['New','Active','At-risk','Churned','Win-back']
PERSONAS=['P1','P2','P3','P4','P5','P6']
OUT=Path('/kaggle/working/persona_lifecycle_outputs' if Path('/kaggle/working').exists() else './persona_lifecycle_outputs')
OUT.mkdir(parents=True,exist_ok=True)
SEARCH=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
def resolve(configured,pattern):
    base=Path(configured) if configured else SEARCH
    roots=sorted({p.parent for p in base.rglob(pattern)})
    if len(roots)!=1:raise ValueError(f'{pattern}: chọn folder duy nhất trong {roots}')
    return roots[0]
T=resolve(TREE_ROOT,'personas_tree_????-??.parquet')
L=resolve(LIFECYCLE_ROOT,'lifecycle_????-??.parquet')
def shift(m,j):return str(pd.Period(m,'M')+j)
def b(x):return x.astype('boolean').to_numpy(dtype=bool,na_value=False)
def safe_ratio(a,c):return a/c.replace(0,np.nan)
def load_known(m):
    d=pd.read_parquet(L/f'lifecycle_{m}.parquet')
    required=['customer_id','month_idx','is_observed','lifecycle','acquisition_pair_valid','has_acquisition_month']
    assert set(required)<=set(d),f'05 thiếu {set(required)-set(d)}'
    assert d.customer_id.notna().all() and d.customer_id.is_unique
    period=pd.Period(m,'M');assert d.month_idx.eq(period.year*12+period.month).all()
    assert d.lifecycle.isin(STATES).all() and d.is_observed.isin([True,False]).all()
    assert not (b(d.acquisition_pair_valid)&d.has_acquisition_month.isna().to_numpy()).any()
    return d

def joined_month(m):
    life=load_known(m)
    p=pd.read_parquet(T/f'personas_tree_{m}.parquet')
    assert p.customer_id.notna().all() and p.customer_id.is_unique
    assert p.as_of_month.astype(str).eq(m).all()
    assert p.persona_id.isin(PERSONAS).all()
    assert set(p.customer_id)<=set(life.customer_id),'Nguồn 05/10 khác population'
    # Lifecycle 05 là nguồn authoritative; kiểm tra tag 10 khớp trước khi thay.
    if 'lifecycle' in p:
        check=p[['customer_id','lifecycle']].merge(life[['customer_id','lifecycle']],on='customer_id',validate='one_to_one',suffixes=('_tree','_05'))
        assert check.lifecycle_tree.eq(check.lifecycle_05).all(),'Lifecycle 05/10 không cùng phiên bản'
    # Không nối nhãn source hoặc lifecycle cũ từ 10 để tránh trùng cột.
    cols=[c for c in ['customer_id','persona_id','persona_name','persona_rule','persona_version','R','F_event_months','n_products','valid_pairs','event_history_group'] if c in p]
    d=life.drop(columns=[c for c in cols if c!='customer_id' and c in life]).merge(p[cols],on='customer_id',how='left',validate='one_to_one')
    d['persona_assigned']=d.persona_id.notna()
    assert not (d.persona_assigned&~d.is_observed.eq(True)).any(),'Không dùng persona hiện tại cho khách vắng snapshot'
    d['persona_context']=d.persona_id.astype('string').fillna('NoCurrentPersona')
    d['as_of_month']=m
    d['persona_status']=np.where(d.persona_assigned,'Assigned','NotAssigned')
    # Nếu 10 có coverage export (thường tháng cuối), giữ chính xác lý do loại.
    cov=T/f'persona_coverage_{m}.parquet'
    if cov.exists():
        q=pd.read_parquet(cov,columns=['customer_id','persona_status'])
        assert q.customer_id.is_unique and set(q.customer_id)==set(d.customer_id)
        d=d.drop(columns='persona_status').merge(q,on='customer_id',validate='one_to_one')
        assert d.persona_status.eq('Assigned').eq(d.persona_assigned).all()
    return d


## 1. Cơ cấu hiện tại và selection coverage
Báo tỷ trọng lifecycle trong mỗi persona, và tỷ lệ được gán persona trong từng lifecycle của toàn known population. Hai mẫu số này khác nhau. Nếu một state không có khách đủ điều kiện, không coi đó là bằng chứng state không tồn tại hoặc không quan trọng.

In [ ]:
coverage_rows=[];composition=[]
for m in EVAL_MONTHS+[RECOMMEND_MONTH]:
    d=joined_month(m)
    for state,g in d.groupby('lifecycle'):
        coverage_rows.append(dict(month=m,lifecycle=state,known_n=len(g),assigned_n=int(g.persona_assigned.sum()),observed_n=int(g.is_observed.eq(True).sum())))
    c=d.groupby(['persona_context','lifecycle']).size().rename('n').reset_index();c['month']=m
    composition.append(c)
coverage=pd.DataFrame(coverage_rows)
coverage['assignment_coverage']=coverage.assigned_n/coverage.known_n
coverage.to_csv(OUT/'assignment_coverage_by_lifecycle.csv',index=False)
composition=pd.concat(composition,ignore_index=True)
composition['share_within_persona']=composition.n/composition.groupby(['month','persona_context']).n.transform('sum')
composition.to_csv(OUT/'persona_lifecycle_composition.csv',index=False)
print('Coverage 05/2016: denominator = tất cả known customers trong lifecycle')
display(coverage.loc[coverage.month.eq(RECOMMEND_MONTH)])
latest=joined_month(RECOMMEND_MONTH)
counts=pd.crosstab(latest.persona_context,latest.lifecycle).reindex(index=PERSONAS+['NoCurrentPersona'],columns=STATES,fill_value=0)
shares=counts.div(counts.sum(axis=1).replace(0,np.nan),axis=0)
display(counts);display(shares.round(4))
counts.to_csv(OUT/'latest_counts.csv');shares.to_csv(OUT/'latest_lifecycle_shares.csv')
fig,ax=plt.subplots(figsize=(9,5));shares.fillna(0).plot.bar(stacked=True,ax=ax)
ax.set(ylabel='Tỷ trọng trong từng persona',title=f'Lifecycle trong persona — {RECOMMEND_MONTH}')
plt.xticks(rotation=25);plt.tight_layout();plt.savefig(OUT/'persona_lifecycle_composition.png',bbox_inches='tight');plt.show()
# Ghi rõ lý do loại hiện có, thay vì ép các khách này vào P1.
display(latest.groupby(['lifecycle','persona_status']).size().rename('n').reset_index())


## 2. Acquisition và hiện diện tháng sau
Acquisition chỉ có nhãn khi 05 xác nhận cặp t→t+1 hợp lệ. Khách không đủ cặp: target để thiếu; không coi là 0. Hiện diện tháng sau đo riêng trên toàn source group.

`lift_vs_persona` so cùng persona; `lift_vs_assigned_cohort` so toàn cohort được gán cây; `lift_vs_same_lifecycle` so các persona trong cùng lifecycle (đều giới hạn assigned cohort). Các ô NoCurrentPersona chỉ mô tả coverage và return, không so lift với assigned cohort. Rate theo lifecycle có thể do mix khách/portfolio, không phải hiệu ứng nhân quả.

MIN_N/MIN_POS chỉ là cờ độ tin cậy ứng viên. Không kết luận từ ô New/Win-back rất nhỏ. CI Wilson theo tháng; không gộp các tháng rồi coi mọi dòng khách độc lập.

In [ ]:
def wilson(k,n):
    k=np.asarray(k,float);n=np.asarray(n,float);ok=n>0
    lo=np.full_like(n,np.nan);hi=lo.copy();pp=k[ok]/n[ok];nn=n[ok];z=1.96
    center=(pp+z*z/(2*nn))/(1+z*z/nn);half=z*np.sqrt(pp*(1-pp)/nn+z*z/(4*nn*nn))/(1+z*z/nn)
    lo[ok]=np.maximum(0,center-half);hi[ok]=np.minimum(1,center+half)
    return lo,hi
rows=[];transitions=[];pair_coverage=[]
for m in EVAL_MONTHS:
    source=joined_month(m)
    future=load_known(shift(m,1))
    cols=['customer_id','is_observed','lifecycle','acquisition_pair_valid','has_acquisition_month']
    future=future[cols].rename(columns={c:'next_'+c for c in cols if c!='customer_id'})
    d=source.merge(future,on='customer_id',how='left',validate='one_to_one')
    matched=d.next_lifecycle.notna()
    pair_coverage.append(dict(month=m,source_known_n=len(d),matched_next_known_n=int(matched.sum())))
    d['label_valid']=b(d.next_acquisition_pair_valid)
    d['positive']=d.label_valid&b(d.next_has_acquisition_month)
    d['present_next']=b(d.next_is_observed)
    d['presence_label_valid']=matched
    for (pid,state),g in d.groupby(['persona_context','lifecycle']):
        rows.append(dict(month=m,persona_id=pid,lifecycle=state,n=len(g),label_n=int(g.label_valid.sum()),positive_n=int(g.positive.sum()),
            presence_label_n=int(g.presence_label_valid.sum()),present_next_n=int(g.present_next.sum())))
    t=d.loc[matched].groupby(['persona_context','lifecycle','next_lifecycle']).size().rename('n').reset_index()
    t['source_month']=m;t['target_month']=shift(m,1);transitions.append(t)
report=pd.DataFrame(rows)
report['label_coverage']=report.label_n/report.n
report['acquisition_rate']=safe_ratio(report.positive_n,report.label_n)
report['presence_label_coverage']=report.presence_label_n/report.n
report['present_next_rate']=safe_ratio(report.present_next_n,report.presence_label_n)
report['ci_low'],report['ci_high']=wilson(report.positive_n,report.label_n)
report['weak_evidence']=(report.label_n<MIN_N)|(report.positive_n<MIN_POS)
assigned=report.loc[report.persona_id.ne('NoCurrentPersona')]
pbase=assigned.groupby(['month','persona_id'])[['label_n','positive_n']].sum()
pbase['persona_baseline']=safe_ratio(pbase.positive_n,pbase.label_n)
lbase=assigned.groupby(['month','lifecycle'])[['label_n','positive_n']].sum()
lbase['lifecycle_baseline']=safe_ratio(lbase.positive_n,lbase.label_n)
base=assigned.groupby('month')[['label_n','positive_n']].sum()
base['assigned_baseline']=safe_ratio(base.positive_n,base.label_n)
report=report.merge(pbase[['persona_baseline']],left_on=['month','persona_id'],right_index=True,how='left')
report=report.merge(lbase[['lifecycle_baseline']],left_on=['month','lifecycle'],right_index=True,how='left')
report=report.merge(base[['assigned_baseline']],left_on='month',right_index=True,how='left')
report['lift_vs_persona']=safe_ratio(report.acquisition_rate,report.persona_baseline)
report['lift_vs_same_lifecycle']=safe_ratio(report.acquisition_rate,report.lifecycle_baseline)
report['lift_vs_assigned_cohort']=safe_ratio(report.acquisition_rate,report.assigned_baseline)
report.loc[report.persona_id.eq('NoCurrentPersona'),['lift_vs_same_lifecycle','lift_vs_assigned_cohort','lifecycle_baseline','assigned_baseline']]=np.nan
report.to_csv(OUT/'persona_lifecycle_validation_1m.csv',index=False)
pd.DataFrame(pair_coverage).to_csv(OUT/'future_join_coverage.csv',index=False)
print('Bảng tháng đánh giá cuối; toàn bộ các tháng trong CSV')
display(report.loc[report.month.eq(EVAL_MONTHS[-1])].round(4))
# Khác biệt lifecycle có đủ bằng chứng không? Nếu chỉ Active đủ mẫu thì chưa thể kết luận thêm giá trị.
support=report.loc[report.persona_id.ne('NoCurrentPersona')].groupby(['persona_id','lifecycle']).agg(months=('month','nunique'),strong_months=('weak_evidence',lambda x:int((~x).sum())),min_label_n=('label_n','min'),min_rate=('acquisition_rate','min'),max_rate=('acquisition_rate','max')).reset_index()
support.to_csv(OUT/'lifecycle_comparison_support.csv',index=False);display(support)
for pid,g in support.groupby('persona_id'):
    reliable=g.loc[g.strong_months.ge(3),'lifecycle'].tolist()
    if len(reliable)<2:print(pid,': chưa đủ ≥2 lifecycle có ít nhất 3 tháng đủ support; không kết luận lifecycle giúp tách nhóm.')


## 3. Chuyển lifecycle và hồ sơ để nối CLV/affinity
Chuyển state đo trên persona **tại tháng nguồn**; persona khách ở tháng sau có thể khác. Không mang persona nguồn sang tháng vắng để gọi là persona hiện tại. Tháng có missing follow-up báo trong future_join_coverage.csv.

File `persona_lifecycle_2016-05.parquet` giữ tất cả known customers, persona null khi không đủ điều kiện. `eligible_persona_lifecycle_2016-05.parquet` chỉ chứa assigned cohort để nối CLV và affinity bước sau. Key duy nhất customer_id + as_of_month.

In [ ]:
transitions=pd.concat(transitions,ignore_index=True)
transitions['transition_rate']=transitions.n/transitions.groupby(['source_month','persona_context','lifecycle']).n.transform('sum')
transitions.to_csv(OUT/'lifecycle_transitions_by_source_persona.csv',index=False)
latest.to_parquet(OUT/f'persona_lifecycle_{RECOMMEND_MONTH}.parquet',index=False)
latest.loc[latest.persona_assigned].to_parquet(OUT/f'eligible_persona_lifecycle_{RECOMMEND_MONTH}.parquet',index=False)
assert not latest.duplicated(['customer_id','as_of_month']).any()
cards=[]
for pid,g in latest.loc[latest.persona_assigned].groupby('persona_id'):
    table=g.lifecycle.value_counts();parts=[f'{state}: {count:,} ({count/len(g):.1%})' for state,count in table.items()]
    reliable=support.loc[support.persona_id.eq(pid)&support.strong_months.ge(3),'lifecycle'].tolist()
    conclusion='Chưa đủ bằng chứng so sánh nhiều lifecycle trong persona.' if len(reliable)<2 else 'Có ít nhất 2 lifecycle đủ support; xem rate/lift/CI theo từng tháng, chưa suy ra nhân quả.'
    name=g.persona_name.iloc[0] if 'persona_name' in g else pid
    cards.append(f'## {pid} — {name}\n- Assigned n={len(g):,}.\n- Lifecycle: '+ '; '.join(parts)+f'\n- Đánh giá: {conclusion}\n')
(OUT/'persona_lifecycle_cards.md').write_text('# Persona × Lifecycle — '+RECOMMEND_MONTH+'\n\n'+'\n'.join(cards),encoding='utf-8')
(OUT/'config.json').write_text(json.dumps(dict(tree_root=str(T),lifecycle_root=str(L),eval_months=EVAL_MONTHS,recommend_month=RECOMMEND_MONTH,min_n=MIN_N,min_positive=MIN_POS,persona_refit=False,scope='development; lifecycle context only; future acquisition complete pairs; presence separate'),ensure_ascii=False,indent=2),encoding='utf-8')
print('Outputs:',OUT)


## Đọc kết quả và bước tiếp theo
- Nếu gần như mọi assigned customer đều Active: lifecycle không giúp tách thêm trong cohort hiện tại; dùng lifecycle để quản lý eligibility/coverage. Không cố tạo đủ mọi tổ hợp.
- New/Win-back ít hoặc không có persona có thể do điều kiện lịch sử ≥3 cặp, không phải khách không quan trọng. Giữ một nhánh onboarding/return riêng thay vì ép vào sáu nhóm.
- At-risk/Churned không có persona hiện tại: đo return/presence ở NoCurrentPersona, không áp rate acquisition của cohort hiện diện.
- Nếu nhiều lifecycle đủ mẫu trong cùng persona: so rate và lift_vs_persona theo tháng trước khi quyết định thêm tag hành động.
- Bước sau: CLV-proxy để ưu tiên khách trong từng nhóm; affinity để chọn sản phẩm chưa sở hữu. Giữ cây cố định. Reopen vs lần đầu sở hữu được quan sát vẫn là audit riêng cần bổ sung trước khi gọi P4 là mở rộng danh mục.
